# Разброс ответов T-lite на одно задание: длина, структура, визуал, поиск

До сих пор на каждое задание было по одному ответу (EXP-008, EXP-009), и разброс мы не видели. Здесь — по 8 ответов
на одно и то же задание с одним и тем же промптом (`prompts.py` после EXP-009: два режима, правила про визуал
и `#SOURCE`, таблицы в статьях). Вопросы:

- насколько ответы на одно задание различаются по качеству — есть ли что усиливать RL (GRPO учится на разнице
  внутри группы ответов: если все 8 одинаковы, учиться не на чем);
- бывают ли у модели «хорошие» ответы — длинные, с заключением, с визуалом, без ошибок — хотя бы иногда
  (решение про SFT: если не бывает никогда, RL их не найдёт);
- длина (слайды, слова), структура (план, титульный слайд, заключение, повторы заголовков), визуал (доля слайдов
  с визуалом, типы, условные графики, `#PLOT` в гуманитарных темах, проблемы отрисовки), поиск (вызовы, повторы,
  источники, числа на слайдах с источником — есть ли они в выдаче эпизода).

Задания: 8 validation из EXP-008 и 4 запроса с фактами; с поиском — все 12, без поиска — 8 validation. По 8 ответов:
160 эпизодов, около 4 часов. Порядок — по кругам: круг k — по одному ответу на каждое задание и режим; если прогон
прервётся, у всех заданий будет поровну ответов. T = 0.9, seed = SEED + 1000·круг + номер задания. Ошибка в эпизоде
(например, в разборе таблиц на настоящей разметке) пишется в таблицу и прогон не останавливает.

Сводная оценка эпизода в конце — черновая, только чтобы мерить разброс одним числом: среднее долей (разбор, длина
относительно нужной, слайды с визуалом, слайды с `#NOTES`, заключение, отрисовка без проблем, принятые источники).
Это не награда для RL.

Результаты — `runs/variance/<время>/`: эпизоды (`<режим>_<id>_<круг>.txt`, `_answer.txt`, `.pptx`), `summary.csv`
(после каждого эпизода), `groups.csv` (по заданиям), `tables_check.txt` (проверка таблиц Википедии).

Запуск на ночь — из терминала в `tmux`, чтобы обрыв связи не остановил прогон (`docs/server.md`).

In [1]:
import csv, dataclasses, json, os, re, statistics, sys, time, traceback
from collections import Counter
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5"  # перед запуском проверить nvidia-smi: карта должна быть свободна
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.dsl import parse
from presentation_rl.environment import build_prompt, run_episode
from presentation_rl.model_loader import hf_generator, load_model
from presentation_rl.prompts import system_prompt
from presentation_rl.rendering import build_presentation, pick_theme
from presentation_rl.tasks import PresentationRequest, load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools

print(torch.__version__, torch.cuda.get_device_name(0))
CONTACT = os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache", WikiClient(contact=CONTACT))
OUT = ROOT / "runs" / "variance" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 0
SAMPLES = 8

2.14.1+cu126 Tesla V100-SXM3-32GB


In [2]:
# Таблицы на настоящей разметке Википедии: в разделах должны быть строки «Таблица:» и «Карточка статьи:».
lines = []
for title in ["Нобелевские лауреаты из России и СССР", "Саяно-Шушенская ГЭС"]:
    try:
        article = cache.article("ru", title)
        for section in article.sections:
            if "Таблица:" in section.text or "Карточка статьи:" in section.text:
                lines += [f"== {title} / {section.heading or 'Введение'}", section.text[:1500], ""]
    except Exception:
        lines += [f"== {title}: ошибка", traceback.format_exc()]
(OUT / "tables_check.txt").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines)[:4000])

== Нобелевские лауреаты из России и СССР / Лауреаты — подданные Российской империи
Таблица:
№ | Год | Направление | Лауреат | Обоснование | Примечание
1 | 1904 | Физиология и медицина | Иван Петрович Павлов | «за работу по физиологии пищеварения» | 
2 | 1905 | Литература | Генрик Сенкевич | «за выдающиеся заслуги в области эпоса» | В материалах Нобелевского комитета указан как представитель Польши, не являвшейся в те годы суверенным государством. Родился в Подляшье (Царство Польское), писал на польском языке, подданный Российской империи на момент получения премии.
3 | 1908 | Физиология и медицина | Илья Ильич Мечников | «за труды по иммунитету» | На момент вручения премии работал во Франции.

== Нобелевские лауреаты из России и СССР / Лауреаты — граждане СССР
— серым цветом отмечены не присутствовавшие на награждении и не произносившие нобелевскую речь.

Таблица:
№ | Год | Направление | Лауреат | Обоснование
1 | 1956 | Химия | Николай Николаевич Семёнов | «за исследования в области ме

In [3]:
start = time.time()
model, tokenizer = load_model("t-tech/T-lite-it-2.1", dtype="float16")
generate = hf_generator(model, tokenizer, temperature=0.9)
print(f"загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

/home/ryzhkova/venvs/diplom/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 399/399 [00:07<00:00, 53.52it/s]


загрузка 17 с, память 7.6 ГБ


## Задания

In [4]:
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
REQUESTS = {key: tasks[key].request for key in ["t367", "t369", "t376", "t381", "t366", "t378", "t386", "t389"]}
REQUESTS.update({
    "s1": PresentationRequest("Сделай презентацию о Полтавской битве по материалам Википедии, с указанием источников.",
                              domain="humanities"),
    "s2": PresentationRequest("Презентация о крупнейших ГЭС России: мощность, год ввода в эксплуатацию, река. "
                              "Данные возьми из Википедии.", slides=8, domain="technical"),
    "s3": PresentationRequest("Сделай презентацию о космическом телескопе «Джеймс Уэбб»: когда запущен, какого размера "
                              "зеркало, где находится и что уже открыл.", domain="technical"),
    "s4": PresentationRequest("Презентация о нобелевских лауреатах по литературе из России и СССР: кто, когда и за что "
                              "получил премию.", domain="humanities"),
})
# (режим, задание): с поиском — все 12, без поиска — 8 validation (запросы s1–s4 без поиска не имеют смысла)
RUNS = [("search", key) for key in REQUESTS] + [("plain", key) for key in REQUESTS if key.startswith("t")]
# Сколько слайдов нужно: из запроса; t366 — один слайд и t369 — «больше 8» slide_range не распознаёт; иначе — 10.
TARGET = {"t366": 1, "t369": 9}
DOMAIN = {key: (tasks[key].domain if key in tasks else request.domain) for key, request in REQUESTS.items()}


def target(key):
    found = REQUESTS[key].slide_range()
    return TARGET.get(key) or (found[0] if found else 10)


print(len(RUNS), "пар (режим, задание) ×", SAMPLES, "=", len(RUNS) * SAMPLES, "эпизодов")

20 пар (режим, задание) × 8 = 160 эпизодов


## Метрики эпизода

Длина, структура, визуал и поиск — по разобранной презентации и отчёту сборщика. Числа на слайдах с принятым
`#SOURCE` (два знака и больше: годы, мощности, проценты) ищутся в ответах инструментов этого эпизода: если числа нет
ни в одном ответе, оно почти наверняка не из прочитанного (обратное не доказывает, что число перенесено верно).

In [5]:
VISUAL = ("CHART", "PLOT", "DIAGRAM", "TIMELINE", "IMAGE", "FORMULA", "QUOTE")
CONCLUSION = re.compile(r"заключ|вывод|итог|резюме|подвед|выводы", re.IGNORECASE)
NUMBER = re.compile(r"\d+(?:\.\d+)?")  # в DSL дробная часть — через точку


def numbers(text):
    found = set()
    for raw in NUMBER.findall(text):
        value = raw
        if len(value.replace(".", "")) >= 2:
            found.add(value)
    return found


def in_responses(value, responses):
    variants = {value, value.replace(".", ","), value.rstrip("0").rstrip(".") if "." in value else value}
    if value.isdigit() and len(value) > 3:  # 6400 в тексте может быть «6 400» или «6400»
        variants.add(f"{int(value):,}".replace(",", " "))
        variants.add(f"{int(value):,}".replace(",", " "))
    return any(v in responses for v in variants)


def measure(mode, key, sample, request, episode, deck, report, tools, seconds):
    slides = deck.slides
    n = len(slides)
    words = sum(len(e.data.get("text", "").split()) for s in slides for e in s.elements if e.kind in ("TEXT", "BULLET"))
    kinds = Counter(e.kind for s in slides for e in s.elements if e.kind in VISUAL)
    titles = [s.title.strip().lower() for s in slides if s.title]
    statuses = Counter(item.status for item in report.items)
    sources = [item.status for item in report.items if item.kind == "SOURCE"]
    plan = deck.plan[0] if deck.plan else ""
    calls = [(c["name"], c["argument"]) for c in episode.calls]
    responses = "\n".join(s.text for s in episode.segments if not s.trainable)
    good_slides = {item.slide for item in report.items if item.kind == "SOURCE" and item.status == "ok"}
    slide_numbers = set()
    for index in good_slides:
        slide = slides[index - 1] if index - 1 < n else None
        if slide is not None:
            slide_numbers |= numbers(" ".join([slide.title or ""] + [e.raw for e in slide.elements]))
    grounded = sum(in_responses(v, responses) for v in slide_numbers)
    need = target(key)
    row = {
        "mode": mode, "id": key, "sample": sample, "domain": DOMAIN[key], "finish": episode.finish,
        "error": "", "generated_tokens": episode.generated_tokens, "seconds": round(seconds),
        # длина
        "slides": n, "target": need, "words": words, "words_per_slide": round(words / n, 1) if n else 0,
        # структура
        "plan_items": len([p for p in plan.split("→") if p.strip()]) if plan else 0,
        "title_slide": bool(n and slides[0].title and not slides[0].notes),
        "conclusion": bool(n >= 3 and slides[-1].title and CONCLUSION.search(slides[-1].title)),
        "repeated_titles": len(titles) - len(set(titles)),
        "notes_share": round(sum(bool(s.notes) for s in slides[1:]) / (n - 1), 2) if n > 1 else 0,
        # визуал
        "visual_share": round(sum(any(e.kind in VISUAL for e in s.elements) for s in slides) / n, 2) if n else 0,
        "visuals_per_slide": round(sum(kinds.values()) / n, 2) if n else 0,
        "visual_types": len(kinds),
        **{kind.lower(): kinds.get(kind, 0) for kind in VISUAL},
        "conditional": sum("условн" in e.raw.lower() for s in slides for e in s.elements if e.kind in ("CHART", "PLOT")),
        "sparse": sum(s.sparse for s in report.slides),
        "render_problems": statuses.get("overflow", 0) + statuses.get("text_overflow", 0) + statuses.get("render_error", 0),
        "parse_errors": len(deck.errors), "parse_rate": round(deck.parse_rate, 3),
        # поиск
        "calls": len(calls), "reads": sum(name == "read" for name, _ in calls),
        "repeated_calls": len(calls) - len(set(calls)),
        "sources_ok": sources.count("ok"), "sources_bad": sources.count("bad_source"),
        "slides_with_source": len(good_slides),
        "numbers_on_source_slides": len(slide_numbers), "numbers_in_results": grounded,
    }
    parts = [deck.parse_rate, min(1.0, n / need) if need else 0, row["visual_share"], row["notes_share"] if need > 1 else 1,
             1.0 if row["conclusion"] or need <= 2 else 0.0, 1 - min(1.0, row["render_problems"] / n) if n else 0]
    if mode == "search" and sources:
        parts.append(sources.count("ok") / len(sources))
    row["score"] = round(sum(parts) / len(parts), 3) if n else 0.0
    return row

In [6]:
rows = []


def save_rows():
    fields = list(dict.fromkeys(k for r in rows for k in r))
    with open(OUT / "summary.csv", "w", newline="", encoding="utf-8-sig") as f:  # utf-8-sig — для Excel
        writer = csv.DictWriter(f, fields)
        writer.writeheader()
        writer.writerows(rows)


def run(mode, key, sample, number):
    request = dataclasses.replace(REQUESTS[key], search=(mode == "search"), images=True)
    stem = f"{mode}_{key}_{sample}"
    start = time.time()
    try:
        tools = WikiTools(cache) if request.search else None
        prompt_ids = build_prompt(tokenizer, system_prompt(request), request)
        torch.manual_seed(SEED + 1000 * sample + number)
        episode = run_episode(prompt_ids, tokenizer, generate, tools, max_new_tokens=8000)
        seconds = time.time() - start
        deck = parse(episode.answer)
        report = build_presentation(deck, OUT / f"{stem}.pptx", pick_theme(request.theme, request.text),
                                    sources=tools.source if tools else None, show_sources=request.sources)
        (OUT / f"{stem}.txt").write_text(tokenizer.decode(episode.ids[len(prompt_ids):]), encoding="utf-8")
        (OUT / f"{stem}_answer.txt").write_text(episode.answer, encoding="utf-8")
        row = measure(mode, key, sample, request, episode, deck, report, tools, seconds)
    except Exception:
        row = {"mode": mode, "id": key, "sample": sample, "error": traceback.format_exc()[-1500:],
               "seconds": round(time.time() - start), "score": ""}
    rows.append(row)
    save_rows()
    print(f"{stem}: {row.get('slides', '—')} сл. (нужно {target(key)}), визуал {row.get('visual_share', '—')}, "
          f"заключение {row.get('conclusion', '—')}, ошибок {row.get('parse_errors', '—')}, вызовов {row.get('calls', '—')}, "
          f"источников {row.get('sources_ok', '—')}/{(row.get('sources_ok') or 0) + (row.get('sources_bad') or 0)}, "
          f"оценка {row['score']}, {row['seconds']} с" + (" — ОШИБКА" if row["error"] else ""))

## Прогон: 8 кругов

In [7]:
numbers_of = {key: i for i, key in enumerate(REQUESTS)}
for sample in range(SAMPLES):
    for mode, key in RUNS:
        run(mode, key, sample, numbers_of[key])
print("готово:", len(rows), "эпизодов, ошибок:", sum(bool(r["error"]) for r in rows))

search_t367_0: 10 сл. (нужно 10), визуал 1.0, заключение True, ошибок 3, вызовов 7, источников 0/0, оценка 0.91, 425 с
search_t369_0: 9 сл. (нужно 9), визуал 1.0, заключение False, ошибок 0, вызовов 10, источников 24/24, оценка 0.841, 279 с
search_t376_0: 4 сл. (нужно 10), визуал 1.0, заключение False, ошибок 2, вызовов 4, источников 5/5, оценка 0.762, 121 с
search_t381_0: 1 сл. (нужно 7), визуал 1.0, заключение False, ошибок 0, вызовов 6, источников 2/2, оценка 0.449, 64 с
search_t366_0: 1 сл. (нужно 1), визуал 1.0, заключение False, ошибок 1, вызовов 0, источников 0/0, оценка 0.972, 32 с
search_t378_0: 6 сл. (нужно 10), визуал 1.0, заключение False, ошибок 2, вызовов 4, источников 13/13, оценка 0.77, 162 с
search_t386_0: 6 сл. (нужно 10), визуал 1.0, заключение False, ошибок 2, вызовов 7, источников 6/6, оценка 0.793, 142 с
search_t389_0: 5 сл. (нужно 10), визуал 1.0, заключение False, ошибок 1, вызовов 4, источников 3/3, оценка 0.753, 72 с
search_s1_0: 5 сл. (нужно 10), визуал 1.0, 

## Разброс по заданиям

По каждой паре (режим, задание): среднее и разброс оценки, лучший из 8 минус средний (насколько лучший ответ
лучше обычного — это и усиливает RL), и «хорошая презентация» хотя бы раз из 8 (pass@8) против доли хороших (pass@1).
Хорошая — не меньше 80% нужных слайдов, заключение (если нужно больше двух слайдов), визуал на 80% слайдов,
разбор без ошибок на 95% строк.

In [8]:
def good(r):
    need = r["target"]
    return (r["slides"] >= 0.8 * need and (r["conclusion"] or need <= 2) and r["visual_share"] >= 0.8
            and r["parse_rate"] >= 0.95)


ok_rows = [r for r in rows if not r["error"]]
groups = []
for mode, key in RUNS:
    group = [r for r in ok_rows if r["mode"] == mode and r["id"] == key]
    if not group:
        continue
    scores = [r["score"] for r in group]
    groups.append({
        "mode": mode, "id": key, "n": len(group), "target": group[0]["target"],
        "score_mean": round(statistics.mean(scores), 3), "score_std": round(statistics.pstdev(scores), 3),
        "score_min": min(scores), "score_max": max(scores), "best_minus_mean": round(max(scores) - statistics.mean(scores), 3),
        "slides_min": min(r["slides"] for r in group), "slides_max": max(r["slides"] for r in group),
        "slides_mean": round(statistics.mean(r["slides"] for r in group), 1),
        "words_mean": round(statistics.mean(r["words"] for r in group)),
        "conclusion_share": round(statistics.mean(r["conclusion"] for r in group), 2),
        "visual_share_mean": round(statistics.mean(r["visual_share"] for r in group), 2),
        "visual_types_mean": round(statistics.mean(r["visual_types"] for r in group), 1),
        "conditional": sum(r["conditional"] for r in group), "plot": sum(r["plot"] for r in group),
        "calls_mean": round(statistics.mean(r["calls"] for r in group), 1),
        "with_calls": sum(r["calls"] > 0 for r in group),
        "repeated_calls": sum(r["repeated_calls"] for r in group),
        "sources_ok": sum(r["sources_ok"] for r in group), "sources_bad": sum(r["sources_bad"] for r in group),
        "numbers_in_results": f"{sum(r['numbers_in_results'] for r in group)}/{sum(r['numbers_on_source_slides'] for r in group)}",
        "pass1": round(statistics.mean(good(r) for r in group), 2), "pass8": any(good(r) for r in group),
        "best": f"{mode}_{key}_{max(group, key=lambda r: r['score'])['sample']}",
        "worst": f"{mode}_{key}_{min(group, key=lambda r: r['score'])['sample']}",
    })
with open(OUT / "groups.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, list(groups[0]))
    writer.writeheader()
    writer.writerows(groups)

columns = ["mode", "id", "target", "slides_min", "slides_max", "score_mean", "score_std", "best_minus_mean",
           "conclusion_share", "visual_share_mean", "calls_mean", "sources_ok", "sources_bad", "pass1", "pass8"]
widths = [max(len(c), *(len(str(g[c])) for g in groups)) for c in columns]
print("  ".join(c.ljust(w) for c, w in zip(columns, widths)))
for g in groups:
    print("  ".join(str(g[c]).ljust(w) for c, w in zip(columns, widths)))

mode    id    target  slides_min  slides_max  score_mean  score_std  best_minus_mean  conclusion_share  visual_share_mean  calls_mean  sources_ok  sources_bad  pass1  pass8
search  t367  10      10          10          0.928       0.069      0.072            0.75              0.93               4.8         34          0            0.5    True 
search  t369  9       0           10          0.737       0.296      0.237            0.38              0.85               8           51          8            0.25   True 
search  t376  10      0           5           0.627       0.239      0.135            0                 0.88               6.2         31          2            0      False
search  t381  7       1           7           0.794       0.144      0.206            0.12              0.91               5.1         43          0            0.12   True 
search  t366  1       1           1           0.908       0.07       0.092            0                 1.0                0.8         

In [9]:
# Итог по режимам: средний разброс внутри задания против разброса между заданиями.
for mode in ("search", "plain"):
    part = [g for g in groups if g["mode"] == mode]
    if not part:
        continue
    print(f"{mode}: заданий {len(part)}; средний разброс оценки внутри задания {statistics.mean(g['score_std'] for g in part):.3f}, "
          f"лучший − средний {statistics.mean(g['best_minus_mean'] for g in part):.3f}; "
          f"хорошая хотя бы раз из 8 — {sum(g['pass8'] for g in part)} из {len(part)}, "
          f"доля хороших {statistics.mean(g['pass1'] for g in part):.2f}")
    print("   лучшие и худшие эпизоды:", "; ".join(f"{g['id']}: {g['best']} / {g['worst']}" for g in part))

search: заданий 12; средний разброс оценки внутри задания 0.125, лучший − средний 0.136; хорошая хотя бы раз из 8 — 6 из 12, доля хороших 0.14
   лучшие и худшие эпизоды: t367: search_t367_1 / search_t367_2; t369: search_t369_1 / search_t369_4; t376: search_t376_0 / search_t376_4; t381: search_t381_3 / search_t381_0; t366: search_t366_7 / search_t366_2; t378: search_t378_0 / search_t378_6; t386: search_t386_0 / search_t386_1; t389: search_t389_6 / search_t389_5; s1: search_s1_0 / search_s1_6; s2: search_s2_6 / search_s2_0; s3: search_s3_2 / search_s3_7; s4: search_s4_0 / search_s4_6
plain: заданий 8; средний разброс оценки внутри задания 0.070, лучший − средний 0.107; хорошая хотя бы раз из 8 — 3 из 8, доля хороших 0.16
   лучшие и худшие эпизоды: t367: plain_t367_3 / plain_t367_2; t369: plain_t369_0 / plain_t369_1; t376: plain_t376_3 / plain_t376_7; t381: plain_t381_3 / plain_t381_1; t366: plain_t366_3 / plain_t366_4; t378: plain_t378_6 / plain_t378_0; t386: plain_t386_4 / plain_t386_